In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from jax import jit, grad
import matplotlib.pyplot as plt
import arviz as az
from IPython.display import Markdown, display

from tb_macro.constants import (
    AGE_STRATA,
    ISO3,
    START_TIME,
    END_TIME,
    LOCAL_OUTPUT_PATH,
    OUTPUT_PATH,
    SOLVER_KWARGS,
)
from tb_macro.epi import get_base_model, add_flows_to_model, initialise_pops
from tb_macro.inputs import load_demography, load_fertility, load_who_outcomes
from tb_macro.demography import prepare_pop_data_for_entries
from tb_macro.parameters import BASE_PARAMS, PARAM_BOUNDS
from tb_macro.calibration import make_log_likelihood
from tb_macro.plotting import (
    plot_comp_distributions,
    plot_dynamic_mixing_matrix,
    plot_mixing_target_comparison,
    plot_single_run_comparison,
    plot_age_population_comparison,
    plot_infection_source_props,
    plot_single_age_s_matrix_components,
    plot_mean_contacts_by_age,
)

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA)  # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))
add_flows_to_model(
    epi_model,
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    death_rates,
    tsr,
    death_in_unsucc,
    entry_times,
    entry_rates,
)
initialise_pops(epi_model, disease_state, age_strat, start_apops)

In [ ]:
MODE = "map"  # "base", "optimise", "random", or "map"

CALIB_SOURCE = "remote"
RUN_ID = "20260915T1826Z"
FOLDER = "60118945"
SAMPLE_SEED = 0

In [ ]:
calib_names = [p for p in PARAM_BOUNDS if p != "mixing_dist_sd"]


def vector_to_params(names, x):
    return dict(zip(names, x))


def params_to_vector(names, params):
    return np.array([params[p] for p in names])


def params_to_bounds(names, bounds):
    return [bounds[p] for p in names]


def load_idata():
    if CALIB_SOURCE == "local":
        path = LOCAL_OUTPUT_PATH / f"{RUN_ID}.nc"
    elif CALIB_SOURCE == "remote":
        path = OUTPUT_PATH / FOLDER / f"{RUN_ID}.nc"
    else:
        raise ValueError(f"Unknown CALIB_SOURCE: {CALIB_SOURCE}")
    if not path.exists():
        raise FileNotFoundError(path)
    return az.from_netcdf(path), path


def posterior_params(sample):
    return {k: sample[k].values.item() for k in sample.data_vars}


log_like = make_log_likelihood(
    epi_model,
    disease_state,
    age_strat,
    infect_strat,
    SOLVER_KWARGS,
    who_mort,
    age_weights,
    group_popsize,
    fert_padded,
)

In [ ]:
opt_res = None
sample_info = None

if MODE == "base":
    run_params = dict(BASE_PARAMS)

elif MODE == "optimise":

    @jit
    def opt_cr(x):
        return -log_like(BASE_PARAMS | vector_to_params(calib_names, x))

    jac_opt_cr = grad(opt_cr)
    inits = params_to_vector(calib_names, BASE_PARAMS)
    bounds = params_to_bounds(calib_names, PARAM_BOUNDS)
    opt_res = minimize(
        fun=opt_cr,
        x0=inits,
        jac=jac_opt_cr,
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxls": 50},
    )
    run_params = BASE_PARAMS | vector_to_params(calib_names, opt_res.x)

elif MODE in {"random", "map"}:
    idata, nc_path = load_idata()
    stacked = idata.posterior.stack(sample=("chain", "draw"))
    n_samples = stacked.sizes["sample"]
    if MODE == "random":
        rng = np.random.default_rng(SAMPLE_SEED)
        idx = int(rng.integers(n_samples))
    else:
        lp = idata.sample_stats["lp"].stack(sample=("chain", "draw"))
        idx = int(np.asarray(lp).argmax())
    sample = stacked.isel(sample=idx)
    sample_info = {
        "path": str(nc_path),
        "index": idx,
        "chain": int(stacked.chain.values[idx]),
        "draw": int(stacked.draw.values[idx]),
        "n_samples": n_samples,
    }
    if MODE == "map":
        sample_info["lp"] = float(np.asarray(lp)[idx])
    run_params = BASE_PARAMS | posterior_params(sample)

In [ ]:
results = epi_model.run(run_params, solver_kwargs=SOLVER_KWARGS)

# Population structure

In [ ]:
display(Markdown("## Population distribution, comparison to target, age distribution and compartment distribution"))
plot_comp_distributions(
    results, disease_state, age_strat, infect_strat, clin_strat, 1950.0, END_TIME, group_popsize
)

In [ ]:
display(Markdown("## Population distribution"))
plot_age_population_comparison(results, group_popsize, age_strat, range(1970, 2021, 10))

# Infection source

In [ ]:
plot_infection_source_props(results, 1950.0, END_TIME)

# Mixing

In [ ]:
display(Markdown("## Mixing matrix over time"))
plot_dynamic_mixing_matrix(results["computed_values"]["dynamic_mm"], 1970.0, 10.0, 3)

In [ ]:
display(Markdown("Mixing matrix proximity to target"))
plot_mixing_target_comparison(results)

In [ ]:
display(Markdown("## Mixing matrix components"))
plot_single_age_s_matrix_components(fert_padded, run_params)

In [ ]:
display(Markdown("## Number of contacts by age group"))
plot_mean_contacts_by_age(results)